# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR^2](https://sen.science/doi/10.71728/senscience.qs2f-h81p) dataset using the [`mlcroissant`](https://mlcroissant.readthedocs.io/) library. All schema entities are referenced by their `@id` fields, ensuring reproducibility and clarity. You can use this notebook as a template for any Croissant-packaged dataset in the biomedical or scientific domains.

### Dataset Source
The dataset is defined by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# If you have not installed mlcroissant, uncomment and run:
!pip install --quiet mlcroissant

## 1. Data Loading

We load metadata and records from the Croissant dataset using the `mlcroissant` library. This makes it easy to inspect the schema, records sets, and available fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Croissant dataset schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}\n")

## 2. Data Overview

Examine available record sets in the dataset and their field IDs.

Each record set and field is referenced via its `@id`.

In [ ]:
# List record sets (@id and name) in the dataset
record_sets = list(dataset.record_sets.keys())
print("Available Record Sets (@id):\n--------------------------")
for rs_id in record_sets:
    rs = dataset.record_sets[rs_id]
    print(f"  - {rs_id}  (name: {getattr(rs, 'name', 'N/A')})")

# For the first record set, list its fields and their @id
if record_sets:
    first_rs_id = record_sets[0]
    fields = dataset.record_sets[first_rs_id].fields
    print(f"\nFields for record set {first_rs_id}:")
    for f in fields:
        print(f" - @id: {f['@id']}	name: {f.get('name', 'N/A')}	dataType: {f.get('dataType', 'N/A')}")

## 3. Data Extraction

Extract all data from each record set into a pandas DataFrame.

For this dataset, we use the `@id` of each record set so you can select or process any of them directly.

In [ ]:
dataframes = {}  # Map from record set @id to DataFrame

for record_set_id in record_sets:
    # List of dicts for each record in this record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from record set {record_set_id}")

# Show schema for first record set
if record_sets:
    first_rs_id = record_sets[0]
    print(f"\nColumns for record set {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's filter, normalize, and group data using actual field `@id`s. We'll work with the first available record set and its fields. 

- We use the field's `@id` to reference the column.

In [ ]:
# Pick the first record set and some suitable field @id
if record_sets:
    rs_id = record_sets[0]
    df = dataframes[rs_id]

    # Try to infer a numeric field @id (e.g., Age or interval)
    # You may need to inspect df.columns first manually
    print(f"Columns in {rs_id}:")
    print(df.columns.tolist())

    # Let's attempt to choose a plausible numeric field by heuristics
    # Try 'age' or interval/promenint numeric columns if present
    possible_numeric_fields = [c for c in df.columns if any(x in c.lower() for x in ['age', 'interval', 'years', 'months', 'time'])]

    if possible_numeric_fields:
        numeric_field_id = possible_numeric_fields[0]
        print(f"Using numeric field: {numeric_field_id}")

        # Remove non-numeric values safely (convert to float)
        numeric_vals = pd.to_numeric(df[numeric_field_id], errors='coerce')
        # Use threshold = median
        threshold = numeric_vals.median()
        filtered_df = df[numeric_vals > threshold].copy()  # Filter
        print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (numeric_vals[numeric_vals > threshold] - numeric_vals.mean()) / numeric_vals.std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by plausible categorical field
        group_candidates = [c for c in df.columns if any(x in c.lower() for x in ['sex', 'gender', 'site', 'location'])]
        if group_candidates:
            group_field_id = group_candidates[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"{numeric_field_id}_mean")
            print(f"\nGrouped data by {group_field_id}:")
            print(grouped_df)
        else:
            print("\nNo suitable categorical field found for grouping.")
    else:
        print("No numeric field detected in columns.")
else:
    print("No record sets found in this Croissant dataset.")

## 5. Visualization

Now, let's plot the distribution of the numeric field, and if available, show its distribution per group (e.g., by sex or anatomical location).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and possible_numeric_fields:
    fig, ax = plt.subplots(figsize=(7,4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce'), kde=True, bins=10, ax=ax)
    ax.set_title(f"Distribution of {numeric_field_id} (record set {rs_id})")
    ax.set_xlabel(numeric_field_id)
    plt.tight_layout()
    plt.show()

    if group_candidates:
        fig, ax = plt.subplots(figsize=(7,4))
        sns.boxplot(y=numeric_field_id, x=group_field_id, data=df, ax=ax)
        ax.set_ylabel(numeric_field_id)
        ax.set_xlabel(group_field_id)
        ax.set_title(f"{numeric_field_id} by {group_field_id}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion

- This notebook illustrated how to load a Croissant-based biomedical dataset using `mlcroissant`, referencing all entities by their `@id`.
- We explored the available record sets and fields, extracted tabular data, and performed simple EDA and visualization on the primary numeric field.
- You can extend this workflow to any dataset with a Croissant schema; just substitute the schema URL and appropriate `@id` values.

**Key next steps:** Consider further analysis, machine learning, or integration with domain knowledge to enable data-driven insights for research or clinical translation.